# SpatialData Vectorize Probe

Small notebook for manually loading the same core inputs used by the pipeline and timing the native `spatialdata` / `sopa` steps.

Keep this shallow:
- load full-merge image
- load cell/nuclear masks
- build a minimal `SpatialData`
- run `spatialdata.to_polygons(...)`
- optionally run `sopa.aggregate(...)` on the vector shapes


In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter

import pandas as pd
import sopa
import spatialdata
import tifffile
from spatialdata import SpatialData
from spatialdata.models import Labels2DModel, ShapesModel
from spatialdata.transformations import Scale, set_transformation


In [ ]:
# Edit these paths manually.
FULL_MERGE_PATH = Path("/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/outputs/SLIDE-0329_full_merge.ome.tif")
CELL_MASK_PATH = Path("/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/outputs/masks/SLIDE-0329_whole_cell.tiff")
NUCLEAR_MASK_PATH = Path("/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/outputs/masks/SLIDE-0329_nuclear.tiff")
NIMBUS_TABLE_PATH = Path("/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/outputs/nimbus/cell_table_full.csv")
PIXEL_SIZE_UM = 0.325

RUN_NUCLEAR = True
RUN_VECTORIZE = True
RUN_SOPA_SHAPES_VECTORIZE = True
RUN_SOPA_VECTOR_AGG = True
LOAD_NIMBUS = False

for path in [FULL_MERGE_PATH, CELL_MASK_PATH]:
    print(path, "exists=", path.exists())
print(NUCLEAR_MASK_PATH, "exists=", NUCLEAR_MASK_PATH.exists())
print(NIMBUS_TABLE_PATH, "exists=", NIMBUS_TABLE_PATH.exists())


In [ ]:
started = perf_counter()
print(f"[probe] loading image: {FULL_MERGE_PATH}")
image_sdata = sopa.io.ome_tif(FULL_MERGE_PATH)
image_key = next(iter(image_sdata.images.keys()))
full_image = image_sdata.images[image_key]
scale0 = full_image["scale0"]["image"]
image_canvas = tuple(int(v) for v in scale0.shape[-2:])
print("image key:", image_key)
print("image canvas:", image_canvas)
print("image load seconds:", round(perf_counter() - started, 2))


In [ ]:
started = perf_counter()
print(f"[probe] loading cell mask: {CELL_MASK_PATH}")
cell_mask = tifffile.imread(CELL_MASK_PATH)
print("cell mask shape:", cell_mask.shape, "dtype:", cell_mask.dtype)

labels = {
    "cell_labels": Labels2DModel.parse(cell_mask, dims=("y", "x")),
}

if RUN_NUCLEAR and NUCLEAR_MASK_PATH.exists():
    print(f"[probe] loading nuclear mask: {NUCLEAR_MASK_PATH}")
    nuclear_mask = tifffile.imread(NUCLEAR_MASK_PATH)
    print("nuclear mask shape:", nuclear_mask.shape, "dtype:", nuclear_mask.dtype)
    labels["nuclear_labels"] = Labels2DModel.parse(nuclear_mask, dims=("y", "x"))

sdata = SpatialData(images={"full_image": full_image}, labels=labels)
set_transformation(sdata.images["full_image"], Scale([PIXEL_SIZE_UM, PIXEL_SIZE_UM], axes=("x", "y")), to_coordinate_system="global")
for label_name in list(sdata.labels.keys()):
    set_transformation(sdata.labels[label_name], Scale([PIXEL_SIZE_UM, PIXEL_SIZE_UM], axes=("x", "y")), to_coordinate_system="global")

print("labels:", list(sdata.labels.keys()))
print("mask + sdata build seconds:", round(perf_counter() - started, 2))


In [ ]:
vectorization_results = {}

if RUN_VECTORIZE:
    for label_name, shape_name in [("cell_labels", "cell_boundaries"), ("nuclear_labels", "nuclear_boundaries")]:
        if label_name not in sdata.labels:
            continue
        print(f"[probe] vectorizing {label_name} -> {shape_name}")
        started = perf_counter()
        polygon_df = spatialdata.to_polygons(sdata.labels[label_name]).copy()
        polygon_df["cell_id"] = polygon_df["label"].astype(int)
        polygon_df["instance_id"] = polygon_df["cell_id"].astype(str)
        polygon_df = polygon_df.sort_values("cell_id").reset_index(drop=True)
        shape_df = polygon_df.set_index("instance_id", drop=False).copy()
        sdata[shape_name] = ShapesModel.parse(shape_df)
        elapsed = perf_counter() - started
        vectorization_results[shape_name] = {
            "rows": int(len(shape_df.index)),
            "seconds": round(elapsed, 2),
        }
        print(vectorization_results[shape_name])

vectorization_results


In [ ]:
sopa_shapes_vectorize_results = {}

if RUN_SOPA_SHAPES_VECTORIZE:
    for label_name, mask_array in [("cell_labels", cell_mask), ("nuclear_labels", nuclear_mask if "nuclear_mask" in globals() else None)]:
        if mask_array is None:
            continue
        print(f"[probe] sopa.shapes.vectorize on {label_name}")
        started = perf_counter()
        geo_df = sopa.shapes.vectorize(mask_array)
        elapsed = perf_counter() - started
        sopa_shapes_vectorize_results[label_name] = {
            "rows": int(len(geo_df.index)),
            "seconds": round(elapsed, 2),
            "columns": list(geo_df.columns),
        }
        print(sopa_shapes_vectorize_results[label_name])

sopa_shapes_vectorize_results


In [ ]:
sopa_vector_agg_results = {}

if RUN_SOPA_VECTOR_AGG:
    for shape_name in ["cell_boundaries", "nuclear_boundaries"]:
        if shape_name not in sdata.shapes:
            continue
        table_name = f"{shape_name}_channels"
        print(f"[probe] sopa.aggregate on {shape_name} -> {table_name}")
        started = perf_counter()
        temp_sdata = SpatialData(images={"full_image": sdata.images["full_image"]}, shapes={shape_name: sdata.shapes[shape_name].copy()})
        sopa.aggregate(
            temp_sdata,
            aggregate_genes=False,
            aggregate_channels=True,
            image_key="full_image",
            shapes_key=shape_name,
            key_added=table_name,
            min_intensity_ratio=0.0,
        )
        elapsed = perf_counter() - started
        sopa_vector_agg_results[table_name] = {
            "rows": int(temp_sdata.tables[table_name].n_obs),
            "vars": int(temp_sdata.tables[table_name].n_vars),
            "seconds": round(elapsed, 2),
        }
        print(sopa_vector_agg_results[table_name])

sopa_vector_agg_results


In [ ]:
if LOAD_NIMBUS and NIMBUS_TABLE_PATH.exists():
    print(f"[probe] loading nimbus table: {NIMBUS_TABLE_PATH}")
    nimbus_df = pd.read_csv(NIMBUS_TABLE_PATH)
    print("nimbus shape:", nimbus_df.shape)
    print("nimbus columns:", list(nimbus_df.columns[:10]))
    display(nimbus_df.head())
else:
    print("Nimbus load skipped.")


In [ ]:
print("images:", list(sdata.images.keys()))
print("labels:", list(sdata.labels.keys()))
print("shapes:", list(sdata.shapes.keys()))
print("tables:", list(sdata.tables.keys()))
sdata
